# Contract Comparison

**Research question:** Given a constrained set of deals, which of two competing contracts scores better on average, and by how much?

This is a minimal template for double-dummy contract comparisons: constrain a couple of seats, pick a contract for each side of the comparison (which can be a fixed contract or a function of the deal), solve both with DDS, and report the average advantage in IMPs, matchpoints, or raw score.

## 1. Set up the deal constraints and the two contracts to compare

North is constrained to a balanced 10-14 HCP hand (excluding 5-card majors/minors so it's a believable 1NT-range hand); South is a fixed hand. `con1`/`con2` take a `Deal` and return a contract string — here they're constants, but they could just as easily inspect `deal.north`/`deal.south` to choose a contract conditionally (e.g. based on shape or a fit).

In [ ]:
import bridgepandas as bp
m = bp.h

east = None
west = None
north = (m.HCP >= 10) & (m.HCP <= 14) & m.MATCH_SHAPE("any 5332 + any 4432 + any 4333 + 2452 + 2425 - 5xxx - 4xxx - x5xx")
south = "AJT3/94/T4/Q9753"

vuln = bp.TableVuln("ns")  # ns, ew, both, none
score_type = "imps"  # imps, matchpoints, total

def con1(deal):
    return "2C-S"

def con2(deal):
    return "1N-N"

## 2. Generate deals

`north` is a `HandSet`, so `random_deals` uses fast BDD sampling rather than accept/reject.

In [ ]:
deals = bp.random_deals(5000, north=north, south=south)

## 3. Apply con1/con2 per deal, solve double-dummy, and report the result

`con1`/`con2` take a `Deal`, not a DataFrame row, so they're applied row-wise (`axis=1`) with each row wrapped in `bp.Deal`. `add_dds_score` then solves both contract columns at once via `columns=["con1", "con2"]`, producing `con1_score`/`con2_score`. The mean and standard error of the mean (`.sem()`) of the score difference give the average advantage and its uncertainty; `diffing_func`/`format_func` convert and format that difference according to `score_type`.

In [ ]:
deals["con1"] = deals.apply(lambda row: con1(bp.Deal(row)), axis=1)
deals["con2"] = deals.apply(lambda row: con2(bp.Deal(row)), axis=1)

diffing_func = {
    "imps": bp.scorediff_imps,
    "matchpoints": bp.scorediff_matchpoints,
    "total": lambda x: x,
}

format_func = {
    "imps":        lambda x: f"{x:5.2f}",
    "matchpoints": lambda x: f"{x*100:5.2f}",
    "total":       lambda x: f"{x:5.1f}",
}

bp.add_dds_score(deals, columns=["con1", "con2"], vuln=vuln, processes=5)
deals["delta"] = diffing_func[score_type](deals.con1_score - deals.con2_score)

ff = format_func[score_type]
print(f"The advantage for con1 is {ff(deals.delta.mean())} +/- {ff(deals.delta.sem())}")